In [8]:
import pandas as pd 

df = pd.read_csv("APL_Logistics.csv", encoding="latin1" )
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Product Name,Product Price,Shipping Mode
0,DEBIT,6,4,159.69,472.45,Late delivery,1,9,Cardio Equipment,Brownsville,...,5,499.95,472.45,159.69,South Asia,Maharashtra,COMPLETE,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class
1,DEBIT,4,4,48.71,167.96,Shipping on time,0,29,Shop By Sport,Littleton,...,5,199.95,167.96,48.71,Central America,Cortés,ON_HOLD,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class
2,DEBIT,4,4,87.36,181.99,Shipping on time,0,48,Water Sports,Littleton,...,1,199.99,181.99,87.36,Central America,Cortés,ON_HOLD,Pelican Sunstream 100 Kayak,199.99,Standard Class
3,DEBIT,6,4,-41.89,175.99,Late delivery,1,48,Water Sports,Littleton,...,1,199.99,175.99,-41.89,East of USA,Nueva York,COMPLETE,Pelican Sunstream 100 Kayak,199.99,Standard Class
4,DEBIT,6,4,10.00,40.00,Late delivery,1,24,Women's Apparel,Littleton,...,1,50.00,40.00,10.00,East of USA,Nueva York,COMPLETE,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class


In [9]:
df.columns = df.columns.str.strip()
print("Shape:", df.shape)
df.head()

Shape: (180519, 40)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Product Name,Product Price,Shipping Mode
0,DEBIT,6,4,159.69,472.45,Late delivery,1,9,Cardio Equipment,Brownsville,...,5,499.95,472.45,159.69,South Asia,Maharashtra,COMPLETE,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class
1,DEBIT,4,4,48.71,167.96,Shipping on time,0,29,Shop By Sport,Littleton,...,5,199.95,167.96,48.71,Central America,Cortés,ON_HOLD,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class
2,DEBIT,4,4,87.36,181.99,Shipping on time,0,48,Water Sports,Littleton,...,1,199.99,181.99,87.36,Central America,Cortés,ON_HOLD,Pelican Sunstream 100 Kayak,199.99,Standard Class
3,DEBIT,6,4,-41.89,175.99,Late delivery,1,48,Water Sports,Littleton,...,1,199.99,175.99,-41.89,East of USA,Nueva York,COMPLETE,Pelican Sunstream 100 Kayak,199.99,Standard Class
4,DEBIT,6,4,10.00,40.00,Late delivery,1,24,Women's Apparel,Littleton,...,1,50.00,40.00,10.00,East of USA,Nueva York,COMPLETE,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class


In [11]:
df.isnull().sum()[df.isnull().sum() > 0]

Customer Lname      8
Customer Zipcode    3
dtype: int64

In [12]:
# Convert to numeric, coerce errors to NaN
df["Days for shipping (real)"] = pd.to_numeric(df["Days for shipping (real)"], errors="coerce")
df["Days for shipment (scheduled)"] = pd.to_numeric(df["Days for shipment (scheduled)"], errors="coerce")

# Check for invalid values
invalid = df[
    (df["Days for shipping (real)"].isna()) |
    (df["Days for shipment (scheduled)"].isna()) |
    (df["Days for shipping (real)"] < 0) |
    (df["Days for shipment (scheduled)"] < 0)
]
print("Invalid shipping duration rows:", len(invalid))

Invalid shipping duration rows: 0


In [13]:
before = df.shape[0]

df = df[
    (df["Days for shipping (real)"].notna()) &
    (df["Days for shipment (scheduled)"].notna()) &
    (df["Days for shipping (real)"] >= 0) &
    (df["Days for shipment (scheduled)"] >= 0)
]

print(f"Removed {before - df.shape[0]} rows. New shape: {df.shape}")

Removed 0 rows. New shape: (180519, 40)


In [14]:
critical_cols = [
    "Delivery Status", "Late_delivery_risk", "Shipping Mode",
    "Order Region", "Order Country", "Market", "Customer Segment"
]

before = df.shape[0]
df = df.dropna(subset=critical_cols)
df = df.drop_duplicates()

print(f"Rows after removing missing/duplicate critical records: {df.shape[0]} (removed {before - df.shape[0]})")

Rows after removing missing/duplicate critical records: 180519 (removed 0)


In [15]:
df["Is_Cancelled"] = df["Delivery Status"] == "Shipping canceled"
print(df["Is_Cancelled"].value_counts())

Is_Cancelled
False    172765
True       7754
Name: count, dtype: int64


In [16]:
text_cols = [
    "Market", "Order Region", "Order Country", "Order State", "Order City",
    "Customer Country", "Customer State", "Customer City",
    "Customer Segment", "Shipping Mode", "Delivery Status"
]

ACRONYM_FIX = {"Usca": "USCA", "Latam": "LATAM", "Usa": "USA", "Us": "US"}

for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.replace(r"\s+", " ", regex=True).str.title()
    for wrong, right in ACRONYM_FIX.items():
        df[col] = df[col].str.replace(rf"\b{wrong}\b", right, regex=True)

print(sorted(df["Market"].unique()))
print(sorted(df["Order Region"].unique()))

['Africa', 'Europe', 'LATAM', 'Pacific Asia', 'USCA']
['Canada', 'Caribbean', 'Central Africa', 'Central America', 'Central Asia', 'East Africa', 'East Of USA', 'Eastern Asia', 'Eastern Europe', 'North Africa', 'Northern Europe', 'Oceania', 'South America', 'South Asia', 'South Of USA', 'Southeast Asia', 'Southern Africa', 'Southern Europe', 'US Center', 'West Africa', 'West Asia', 'West Of USA', 'Western Europe']


In [17]:
df.to_csv("step1_cleaned_data.csv", index=False)
print("Saved! Final shape:", df.shape)

Saved! Final shape: (180519, 41)
